In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from fastai.vision.all import *
from fastai.data.transforms import IndexSplitter

from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.feature_selection import mutual_info_regression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

import torch.nn as nn

def evaluate_real_price(y_true_log, y_pred_log, name="Model"):
    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)
    
    rmse = root_mean_squared_error(y_true_real, y_pred_real)
    mae = mean_absolute_error(y_true_real, y_pred_real)
    r2 = r2_score(y_true_real, y_pred_real)
    mape = np.mean(np.abs((y_true_real - y_pred_real)/(y_true_real + 1e-9)))*100
    
    print(f"Results for {name}:")
    print(f"  RMSE: {rmse:,.2f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R²:   {r2:.3f}")
    
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape, "R2": r2}

def train_model_with_transfer_learning(learn, model_name, head_epochs=5, ft_epochs=5, head_lr=1e-3, show_lr_plot=True):
    """
    A helper function to standardize the transfer learning process.
    
    Args:
        learn: The FastAI learner object.
        model_name (str): Base name for saving the model.
        head_epochs (int): Number of epochs to train the head.
        ft_epochs (int): Number of epochs for fine-tuning.
        head_lr (float): Learning rate for head training.
        show_lr_plot (bool): Whether to show the LR find plot.
    """
    learn.freeze()
    learn.fit_one_cycle(head_epochs, head_lr)
    lrs = learn.lr_find()
    if(show_lr_plot):
        print(lrs)
    learn.unfreeze()
    learn.fit_one_cycle(ft_epochs, slice(lrs.valley/10, lrs.valley))
    learn.save(f'{model_name}')
    return learn

def extract_embeddings(learner, dl, device=None, pca_dim=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    learner.model.eval()
    learner.model = learner.model.to(device)
    body = learner.model[0].to(device)

    embs = []
    with torch.no_grad():
        for xb in dl:
            x = xb[0] if isinstance(xb, (tuple, list)) else xb
            x = x.to(device)
            feat = body(x)
            pooled = torch.nn.functional.adaptive_avg_pool2d(feat,1).squeeze(-1).squeeze(-1)
            embs.append(pooled.cpu().numpy())
    embs = np.vstack(embs)

    if pca_dim is not None and pca_dim < embs.shape[1]:
        pca = PCA(n_components=pca_dim, random_state=42)
        embs = pca.fit_transform(embs)

    return embs, pca

In [11]:
data_csv = Path("Ad_table_with_files.csv")
df = pd.read_csv(data_csv)

/tmp/ipykernel_3012/52386573.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_csv)


In [12]:
int_cols = ['Reg_year', 'Seat_num', 'Door_num', 'Adv_month', 'Price', 'Runned_Miles']

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    median_val = int(df[col].median())
    df[col] = df[col].fillna(median_val).astype(int)


df['Bodytype'] = df['Bodytype'].fillna(df['Bodytype'].mode()[0])
df['Gearbox'] = df['Gearbox'].fillna(df['Gearbox'].mode()[0])
df['Fuel_type'] = df['Fuel_type'].fillna(df['Fuel_type'].mode()[0])

df_encoded = pd.get_dummies(df, columns=['Bodytype','Gearbox', 'Fuel_type'])

In [13]:
df_encoded.describe(include='all')

,Maker,Genmodel,Genmodel_ID,Adv_ID,Adv_year,Adv_month,Color,Reg_year,Runned_Miles,Engin_size,...,Gearbox_Semi-Automatic,Fuel_type_Bi Fuel,Fuel_type_Diesel,Fuel_type_Electric,Fuel_type_Hybrid Diesel/Electric,Fuel_type_Hybrid Diesel/Electric Plug-in,Fuel_type_Hybrid Petrol/Electric,Fuel_type_Hybrid Petrol/Electric Plug-in,Fuel_type_Petrol,Fuel_type_Petrol Ethanol
count,56249,56249,56249,56249,56249.000000,56249.000000,51491,56249.000000,56249.000000,55564,...,56249,56249,56249,56249,56249,56249,56249,56249,56249,56249
unique,63,732,717,56249,NaN,NaN,21,NaN,NaN,64,...,2,2,2,2,2,2,2,2,2,2
top,Audi,Juke,64_14,10_1$$8,NaN,NaN,Black,NaN,NaN,2.0L,...,False,False,True,False,False,False,False,False,False,False
freq,9349,1646,1646,1,NaN,NaN,12185,NaN,NaN,14954,...,56218,56248,33324,55742,56201,56229,55392,55678,35329,56248
mean,NaN,NaN,NaN,NaN,2017.936230,5.650145,NaN,2013.665434,42385.163416,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,0.262582,2.208549,NaN,3.020549,33949.121235,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,2012.000000,1.000000,NaN,2000.000000,2.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,2018.000000,4.000000,NaN,2012.000000,15235.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,2018.000000,5.000000,NaN,2014.000000,33661.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,2018.000000,7.000000,NaN,2016.000000,62000.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df_encoded['LogPrice'] = np.log1p(df_encoded['Price'])
mu, sigma = df_encoded['LogPrice'].mean(), df_encoded['LogPrice'].std()
df_encoded['NormPrice'] = (df_encoded['LogPrice'] - mu) / sigma

#STRATIFIED SPLITTER
df_encoded = df_encoded.reset_index(drop=True)
df_encoded['price_bin'] = pd.qcut(df_encoded['Price'], q=10, duplicates='drop')

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(sss.split(df_encoded, df_encoded['price_bin']))

stratified_splitter = IndexSplitter(valid_idx)

In [17]:
df_encoded['CarAge'] = (df_encoded['Adv_year'] - df_encoded['Reg_year']).clip(lower=0)
df_encoded['MilesPerYear'] = df_encoded['Runned_Miles'] / df_encoded['CarAge'].replace(0,1)
df_encoded['AdvMonth_sin'] = np.sin(2*np.pi*df_encoded['Adv_month']/12)
df_encoded['AdvMonth_cos'] = np.cos(2*np.pi*df_encoded['Adv_month']/12)

cand = ['CarAge','MilesPerYear','Seat_num','Runned_Miles','Reg_year','Adv_month','AdvMonth_sin','AdvMonth_cos']
cat_cols = [c for c in df_encoded.columns if c.startswith('Bodytype_') or c.startswith('Gearbox_') or c.startswith('Fuel_type_')]
all_features = cand + cat_cols
y_log = df_encoded['LogPrice']

spearman = {c: df_encoded[c].corr(y_log, method='spearman') for c in all_features}
print('Spearman corr with log(Price):')
print(pd.Series(spearman).sort_values(key=np.abs, ascending=False))

X_all = df_encoded[all_features].astype(float).values
mi = mutual_info_regression(X_all, y_log, random_state=42)
print('\nMutual information with log(Price):')
print(pd.Series(mi, index=all_features).sort_values(ascending=False))

Spearman corr with log(Price):
CarAge                                      -0.597492
Gearbox_Manual                              -0.594334
Reg_year                                     0.593356
Gearbox_Automatic                            0.592978
Runned_Miles                                -0.541321
Bodytype_Hatchback                          -0.447012
Bodytype_SUV                                 0.336574
Bodytype_Coupe                               0.195246
Bodytype_MPV                                -0.177020
Fuel_type_Petrol                            -0.151515
MilesPerYear                                -0.108449
Fuel_type_Diesel                             0.108220
Fuel_type_Hybrid  Petrol/Electric Plug-in    0.105633
Bodytype_Saloon                              0.089519
AdvMonth_cos                                 0.069926
Bodytype_Convertible                         0.067189
Fuel_type_Hybrid  Petrol/Electric            0.053981
AdvMonth_sin                                 0.0424

In [ ]:
dblock = DataBlock(
    blocks = (ImageBlock, RegressionBlock),
    get_x = ColReader('FilePath'),
    get_y = ColReader('LogPrice'),
    splitter = stratified_splitter,                                    #Using the stratified splitter now!
    batch_tfms = Normalize.from_stats(*imagenet_stats)
)

dls_log_stratified = dblock.dataloaders(df_encoded, bs=32)

baseline_visual = vision_learner(
    dls_log_stratified,
    resnet34,
    loss_func=MSELossFlat(),
    metrics=rmse
)

baseline_visual = train_model_with_transfer_learning(baseline_visual, 'car-price-log-stratified')

epoch,train_loss,valid_loss,_rmse,time
0,4.480149,1.488544,1.220059,01:31
1,0.886552,0.329053,0.573631,01:31
2,0.628052,0.220207,0.469263,01:31
3,0.507763,0.163575,0.404444,01:29
4,0.464555,0.155863,0.394796,01:32


SuggestedLRs(valley=0.00019054606673307717)


epoch,train_loss,valid_loss,_rmse,time
0,0.513748,0.172675,0.415542,02:04
1,0.427166,0.132334,0.363777,02:05
2,0.399516,0.101411,0.318452,02:03
3,0.350468,0.083616,0.289165,02:07


In [22]:
img_embs, pca = extract_embeddings(baseline_visual, dls_log_stratified.test_dl(df_encoded), pca_dim=100)
tab_feats = [
    'CarAge', 'Runned_Miles', 'Seat_num',
    'MilesPerYear', 'AdvMonth_sin', 'AdvMonth_cos',
    'Gearbox_Automatic', 'Gearbox_Manual',
    'Bodytype_Hatchback', 'Bodytype_SUV', 'Bodytype_Coupe', 'Bodytype_MPV',
    'Fuel_type_Petrol', 'Fuel_type_Diesel'
]

X_tab = df_encoded[tab_feats].astype(float).values

X_combined = np.hstack([X_tab, img_embs])

img_cols = [f"img_{i}" for i in range(img_embs.shape[1])]
X_df = pd.DataFrame(X_combined, columns=tab_feats + img_cols)

y = df_encoded['LogPrice'].values

X_train, X_valid, y_train, y_valid = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

In [23]:
#Random Forest Regressor
model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train,y_train)
preds = model.predict(X_valid)

metrics = evaluate_real_price(y_valid, preds, name="Tabular+Embeddings_Forest")

Results for Tabular+Embeddings_Forest:
  RMSE: 25,130.02
  MAE:  3,207.14
  MAPE: 16.00%
  R²:   0.466


In [ ]:
joblib.dump(pca, "pca.pkl")
joblib.dump(model, "forest_model.pkl")

baseline_visual.export("vision_model.pkl")

In [28]:
# Tabular only
metrics_tab = evaluate_real_price(y_valid, 
                                  RandomForestRegressor(n_estimators=300).fit(X_train[:, :len(tab_feats)], y_train)
                                  .predict(X_valid[:, :len(tab_feats)]), 
                                  name="Tabular Only")

# Image embeddings only
metrics_img = evaluate_real_price(y_valid, 
                                  RandomForestRegressor(n_estimators=300).fit(X_train[:, len(tab_feats):], y_train)
                                  .predict(X_valid[:, len(tab_feats):]), 
                                  name="Image Only")

Results for Tabular Only:
  RMSE: 30,632.94
  MAE:  6,282.78
  MAPE: 32.31%
  R²:   0.206
Results for Image Only:
  RMSE: 25,508.82
  MAE:  3,610.67
  MAPE: 19.23%
  R²:   0.449
